# Datasheet PDF → Qdrant ingestion for agentic RAG

Ingestion pipeline tuned for **semiconductor datasheets**: dense spec tables, typical-performance graphs, pinouts, and application circuits.

| Stage | What happens | Why |
|---|---|---|
| **1. Vision parsing** | Each page is rendered to an image and sent with its PDF text layer to a vision LLM (`gpt-5.5`), which writes structured Markdown | Plain text extraction scrambles MIN/TYP/MAX columns and merges stacked rows. The vision model rebuilds the table structure, and the text layer keeps the numbers exact. Graphs and circuits are turned into searchable text descriptions. |
| **2. Parse cache** | Page Markdown is saved to `parsed/` | Re-runs don't pay for parsing again, and you can inspect or fix the Markdown by hand |
| **3. Structure-aware chunking** | Prose is split by section, tables by row group (header and test conditions repeated in every chunk), and each figure is one chunk. Full pages are also stored. | Each chunk makes sense on its own. The agent can widen context through table, neighbor, and page lookups. |
| **4. Contextual header** | Every chunk is prefixed with `[part · section · table · page]` | Improves both embedding and keyword search, and gives the LLM citations |
| **5. Hybrid vectors** | Dense `text-embedding-3-large` plus sparse BM25 (Qdrant computes the IDF) | Part numbers (`LTC3310HV-1#PBF`), pin names (`PGOOD`) and symbols need exact keyword matching. Paraphrased questions need semantic matching. |
| **6. Hybrid retrieval** | Dense and sparse candidates combined with Reciprocal Rank Fusion, plus metadata filters | |
| **7. Observability** | Arize Phoenix traces every stage: OpenAI calls (automatic) plus spans for parse, chunk, embed, upsert, and retrieval | See per-page parse output, latency, token cost, failures and fallbacks, and what each search returned |

**Setup** (run once in a terminal in this folder):
```
docker compose up -d     # Qdrant on localhost:6333 (dashboard: /dashboard) + Phoenix on localhost:6006
uv sync                  # creates .venv with all dependencies
```
Then select the `.venv` interpreter as this notebook's kernel. The OpenAI key is read from `.env`.

## 1. Configuration

In [ ]:
import os, re, json, uuid, base64, hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

# shared settings (.env, TLS via OS cert store, Qdrant/Phoenix endpoints, embedding model)
from datasheet_rag import config
from datasheet_rag.config import COLLECTION, EMBED_MODEL, EMBED_DIM, PHOENIX_PROJECT

PDF_PATHS = [
    Path(r"C:\Users\venkatesh.chandragir\Downloads\ltc3310s-3310s-1-3310-3310-1.pdf"),
    *sorted(Path("pdfs").glob("*.pdf")),          # anything else dropped into ./pdfs
]
PARSED_DIR   = Path("parsed")

PARSE_MODEL      = "gpt-5.5"          # vision model for page → Markdown
PARSE_EFFORT     = "low"              # reasoning effort; "medium" = slower, marginally better graph reading
PARSE_DPI        = 200
PARSE_WORKERS    = 6                  # pages parsed concurrently
META_MODEL       = "gpt-5.4-mini"     # document-level metadata extraction
EMBED_BATCH      = 64

TEXT_CHUNK_TOKENS  = 450              # prose chunk budget
TABLE_CHUNK_TOKENS = 600              # table-row budget per chunk (header added on top)
PAGE_MAX_TOKENS    = 7500             # page-level points are truncated to fit the embedding model

TRACE_PAGE_IMAGES  = False            # True = store each rendered page image in its trace (handy for checking parses; ~1 MB/page)

PARSED_DIR.mkdir(exist_ok=True)
for p in PDF_PATHS:
    assert p.exists(), f"Not found: {p}"
print("PDFs:", [p.name for p in PDF_PATHS])

## 1b. Observability: Arize Phoenix

Traces go to the Phoenix container started by `docker compose` (UI at http://localhost:6006, project `datasheet-rag`). The setup lives in `datasheet_rag/tracing.py` and is shared with the agent notebook.

- **Automatic:** `OpenAIInstrumentor` records every OpenAI call (vision parse, metadata, embeddings) with prompts, outputs, token counts, and latency. Embedding vectors are hidden to keep traces small. Page images are redacted unless `TRACE_PAGE_IMAGES = True`.
- **Manual spans** (OpenInference span kinds, so Phoenix shows each one in the right view):
  - `ingest_pdf` (CHAIN) contains `parse_pdf`, one `parse_page` per page, `extract_doc_meta`, `build_chunks`, `embed_chunks`, and `qdrant_upsert`.
  - `search_datasheets` (RETRIEVER) records the query, filters, and retrieved documents with scores.
  - The other retrieval tools are TOOL spans.
- Worker threads inherit the trace context, so concurrent page parses are nested under their PDF's trace.

In [ ]:
from openinference.instrumentation import using_attributes
from opentelemetry.trace import Status, StatusCode
from datasheet_rag.tracing import setup_tracing, tracer, as_json, submit_traced

tracer_provider = setup_tracing(PHOENIX_PROJECT, trace_images=TRACE_PAGE_IMAGES)

## 2. Clients

In [ ]:
import pymupdf
import pymupdf4llm
import tiktoken
from tqdm.auto import tqdm
from qdrant_client import models
from datasheet_rag.clients import oai, qdrant

enc = tiktoken.get_encoding("cl100k_base")       # tokenizer of text-embedding-3-*

def n_tokens(s: str) -> int:
    return len(enc.encode(s))

print("Qdrant collections:", [c.name for c in qdrant.get_collections().collections])

## 3. Vision parsing: page → Markdown

The prompt sets a strict output format that the chunker relies on: `##`/`###` headings, a `Table: <title>` line before every table, one stand-alone row per parameter, and `<figure>` blocks for graphs and drawings.

In [ ]:
PARSE_PROMPT = """You are converting one page of a semiconductor datasheet into clean Markdown for a retrieval system. You get the page image and the PDF text layer. Use the image for structure/layout and the text layer for exact characters (numbers, symbols, part numbers).

Rules:
- Transcribe ALL text content faithfully. Never invent, infer or round values. Keep units and symbols (±, µ, Ω, °C, ≤). Write subscripts inline (VIN, VOUT, TA, RDS(ON)).
- Use '## ' for section titles exactly as printed (e.g. '## ELECTRICAL CHARACTERISTICS', '## PIN FUNCTIONS'). Use '### ' for sub-headings in body text (e.g. '### Inductor Selection'). Do not repeat a section heading just because the page continues it.
- Tables: GitHub Markdown tables. Precede every table with a line 'Table: <exact table title or section name>'; if the table continues from a previous page and has no title on this page, write 'Table: (continued)'. Keep any notes/conditions sentence that belongs to the table directly above the 'Table:' line. Always include the column header row.
  - Every value goes in its correct column (MIN/TYP/MAX/UNITS etc.).
  - Stacked parameters sharing one visual row (e.g. 'Undervoltage Lockout' and 'Undervoltage Lockout Hysteresis') become SEPARATE rows.
  - A cell that spans several rows (shared CONDITIONS, UNITS, PACKAGE, PART MARKING...) is REPEATED in every row it applies to, so every row is self-contained.
  - Group-label rows inside a table (e.g. 'Input Supply', 'Voltage Regulation', or a part-family label) stay as a row with only the first cell filled.
  - If rows are marked with ● (spec applies over full operating temperature range), add a column 'FULL_TEMP' with 'yes' on those rows.
- Dot-leader spec lists (e.g. ABSOLUTE MAXIMUM RATINGS: 'VIN ........ –0.3V to 6V') are tables too: 'Table: <section name>' then a 'PARAMETER | RATING' table with one row per item. For nested items (e.g. 'Operating Junction Temperature Range' followed by per-grade lines) write the parent name into each child row, e.g. 'Operating Junction Temperature Range, LTC3310SE/LTC3310SE-1 | –40°C to 125°C'.
- Pin descriptions ('VIN (Pins 1, 2): ...') stay as paragraphs, one paragraph per pin, starting with the pin name and numbers.
- Output plain Markdown only: no HTML tags or entities (no &nbsp;, <br>), except the <figure> blocks below. Never output a '### ' heading that just repeats the current section title.
- Figures (graphs, plots, block diagrams, application circuits, pinouts, package drawings, timing diagrams) become:
  <figure>
  Title: <exact caption or graph title>
  Type: <graph | block diagram | application circuit | pinout | package drawing | timing diagram | other>
  Description: <for graphs: X axis (name, units, range), Y axis (name, units, range), each curve/legend entry with its conditions, the trend, and key values read from the plot marked 'approximately'. For circuits/block diagrams: every component with its value, how they connect, input/output voltages and currents. For pinouts/packages: every pin number and name, dimensions with units.>
  </figure>
- Drop repeating page furniture: the part-number banner, 'Rev. X', 'For more information www.analog.com', bare page numbers.
- Output only the Markdown. No commentary, no code fences."""

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def parse_page_vision(pdf_path: Path, page_no: int) -> str:
    with pymupdf.open(pdf_path) as doc:            # own handle per thread
        page = doc[page_no - 1]
        png = page.get_pixmap(dpi=PARSE_DPI).tobytes("png")
        text_layer = page.get_text()
    resp = oai.responses.create(
        model=PARSE_MODEL,
        reasoning={"effort": PARSE_EFFORT},
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": f"{PARSE_PROMPT}\n\n<text_layer>\n{text_layer}\n</text_layer>"},
            {"type": "input_image", "image_url": "data:image/png;base64," + base64.b64encode(png).decode(), "detail": "high"},
        ]}],
    )
    return resp.output_text.strip()

def parse_page_fallback(pdf_path: Path, page_no: int) -> str:
    # Non-LLM fallback: PyMuPDF layout-aware Markdown (tables detected heuristically)
    with pymupdf.open(pdf_path) as doc:
        return pymupdf4llm.to_markdown(doc, pages=[page_no - 1], show_progress=False)

def parse_pdf(pdf_path: Path, doc_id: str) -> list[dict]:
    # Returns [{page, markdown, method}], using/filling the on-disk cache.
    # cache key = file content + parser config, so editing the prompt/model triggers a re-parse
    cfg = hashlib.sha256(f"{PARSE_MODEL}|{PARSE_EFFORT}|{PARSE_DPI}|{PARSE_PROMPT}".encode()).hexdigest()[:8]
    cache = PARSED_DIR / f"{pdf_path.stem}__{doc_id[:12]}__{cfg}"
    cache.mkdir(exist_ok=True)
    with pymupdf.open(pdf_path) as doc:
        n_pages = doc.page_count

    def work(page_no):
        with tracer.start_as_current_span(f"parse_page {page_no}", openinference_span_kind="chain") as span:
            span.set_attributes({"pdf.source": pdf_path.name, "pdf.page": page_no})
            span.set_input(f"{pdf_path.name} page {page_no}")
            md_file, meta_file = cache / f"page_{page_no:03d}.md", cache / f"page_{page_no:03d}.json"
            if md_file.exists() and meta_file.exists():
                md_text, method = md_file.read_text(encoding="utf-8"), json.loads(meta_file.read_text())["method"]
                span.set_attribute("parse.cache_hit", True)
            else:
                span.set_attribute("parse.cache_hit", False)
                try:
                    md_text, method = parse_page_vision(pdf_path, page_no), PARSE_MODEL
                except Exception as e:
                    print(f"  ⚠ page {page_no}: vision parse failed ({type(e).__name__}: {e}); using fallback")
                    span.record_exception(e)
                    span.set_attribute("parse.fallback_reason", f"{type(e).__name__}: {e}")
                    md_text, method = parse_page_fallback(pdf_path, page_no), "pymupdf4llm"
                md_file.write_text(md_text, encoding="utf-8")
                meta_file.write_text(json.dumps({"method": method}))
            span.set_attributes({"parse.method": method, "parse.output_chars": len(md_text),
                                 "parse.tables": md_text.count("\nTable:") + md_text.startswith("Table:"),
                                 "parse.figures": md_text.count("<figure>")})
            span.set_output(md_text, mime_type="text/plain")
            return {"page": page_no, "markdown": md_text, "method": method}

    with tracer.start_as_current_span("parse_pdf", openinference_span_kind="chain") as span:
        span.set_input(str(pdf_path))
        span.set_attributes({"pdf.pages": n_pages, "parse.model": PARSE_MODEL, "parse.cache_dir": str(cache)})
        pages = []
        with ThreadPoolExecutor(PARSE_WORKERS) as pool:
            futures = [submit_traced(pool, work, i) for i in range(1, n_pages + 1)]
            for f in tqdm(as_completed(futures), total=n_pages, desc=f"Parsing {pdf_path.name}"):
                pages.append(f.result())
        methods = Counter(p["method"] for p in pages)
        span.set_attributes({f"parse.pages_by_{k}": v for k, v in methods.items()})
        span.set_output(as_json({"pages": n_pages, "methods": methods}), mime_type="application/json")
    return sorted(pages, key=lambda p: p["page"])

## 4. Document-level metadata (part numbers, device type, summary)

In [ ]:
DOC_META_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "required": ["title", "manufacturer", "part_numbers", "device_category", "summary"],
    "properties": {
        "title":           {"type": "string", "description": "Datasheet title, e.g. 'LTC3310S/LTC3310S-1 ... 5V, 10A Synchronous Step-Down Silent Switcher'"},
        "manufacturer":    {"type": "string"},
        "part_numbers":    {"type": "array", "items": {"type": "string"}, "description": "Base part numbers covered (no ordering suffixes)"},
        "device_category": {"type": "string", "description": "e.g. 'synchronous buck converter'"},
        "summary":         {"type": "string", "description": "2-3 sentence technical summary with key specs"},
    },
}

def extract_doc_meta(pages: list[dict]) -> dict:
    head = "\n\n".join(p["markdown"] for p in pages[:2])
    resp = oai.responses.create(
        model=META_MODEL,
        input=f"Extract metadata from the first pages of this datasheet.\n\n{head}",
        text={"format": {"type": "json_schema", "name": "doc_meta", "schema": DOC_META_SCHEMA, "strict": True}},
    )
    return json.loads(resp.output_text)

## 5. Structure-aware chunking

Walks the parsed Markdown of all pages in order and keeps track of the current section, sub-section, table title, table header, and table conditions note across page breaks:
- **Prose:** packed within a sub-section up to `TEXT_CHUNK_TOKENS`, never crossing a heading.
- **Tables:** split at group-label rows (e.g. *Input Supply*, *Voltage Regulation*), packed up to `TABLE_CHUNK_TOKENS`. Every chunk repeats the table title, the conditions note (e.g. *VIN = 3.3V, TA = 25°C unless noted*), and the column header. A table continued on the next page keeps the same `table_id`.
- **Figures:** one chunk each.
- **Pins:** each `NAME (Pin N): …` description is its own `pin` chunk, with a `pin_name` field.
- **Pages:** the full page Markdown is stored as a separate `page` point, for broader context.

In [ ]:
_HEADING = re.compile(r"^(#{1,6})\s+(.*)$")
# 'PGOOD (Pin 14): ...', 'VIN (Pins 3, 4, 11, 12): ...', 'GND (Exposed Pad Pin 19): ...'
_PIN_RE = re.compile(r"^\**([A-Z][A-Za-z0-9/_+\-]*(?:\s[A-Z0-9/_+\-]+)?)\**\s*\(((?:Exposed Pad\s*)?Pins?\b[^)]*)\):")
TABLE_NOTE_MAX_TOKENS = 120     # a paragraph right before a table is kept as its note only if short

def parse_blocks(md_text: str) -> list[tuple]:
    lines, blocks, i = md_text.splitlines(), [], 0
    special = ("|", "#", "<figure>", "Table:")
    while i < len(lines):
        s = lines[i].strip()
        if not s:
            i += 1; continue
        if s.startswith("<figure>"):
            buf = []
            while i < len(lines):
                buf.append(lines[i].strip()); i += 1
                if buf[-1].startswith("</figure>"):
                    break
            blocks.append(("figure", "\n".join(buf))); continue
        if m := _HEADING.match(s):
            blocks.append(("heading", len(m.group(1)), m.group(2).strip().strip("*"))); i += 1; continue
        if s.startswith("Table:"):
            blocks.append(("table_title", s[6:].strip())); i += 1; continue
        if s.startswith("|"):
            rows = []
            while i < len(lines) and lines[i].strip().startswith("|"):
                rows.append(lines[i].strip()); i += 1
            blocks.append(("table", rows)); continue
        buf = []
        while i < len(lines) and lines[i].strip() and not lines[i].strip().startswith(special):
            buf.append(lines[i].strip()); i += 1
        blocks.append(("para", " ".join(buf)))
    return blocks

def _cells(row: str) -> list[str]:
    return [c.strip() for c in row.strip().strip("|").split("|")]

def _is_sep(row: str) -> bool:
    return bool(re.fullmatch(r"\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)*\|?", row.strip()))

def _is_group_row(row: str) -> bool:
    c = _cells(row)
    return len(c) > 1 and bool(c[0]) and not any(c[1:])

def build_chunks(pages: list[dict], meta: dict, source: str) -> list[dict]:
    part = "/".join(meta["part_numbers"][:4]) or meta["title"]
    chunks = []
    st = {"section": None, "subsection": None, "table_title": None, "last_table_title": None, "table_id": None,
          "table_note": None, "pending_note": None, "table_header": None, "last_para": None, "prev_kind": None}
    text_buf: list[tuple[str, int]] = []      # (paragraph, page)
    n_tables = 0

    def header(page_lbl, extra=None):
        path = " > ".join(x for x in (st["section"], st["subsection"]) if x)
        bits = [part, path or None, extra, f"p.{page_lbl}"]
        return "[" + " · ".join(b for b in bits if b) + "]"

    def emit(kind, content, page_start, page_end=None, **extra):
        page_end = page_end or page_start
        lbl = str(page_start) if page_start == page_end else f"{page_start}-{page_end}"
        hdr = header(lbl, f"Table: {extra['table_title']}" if kind == "table" else
                          f"Pin: {extra['pin_name']}" if kind == "pin" else None)
        chunks.append({"content_type": kind, "text": f"{hdr}\n{content}", "content": content,
                       "section": st["section"], "subsection": st["subsection"],
                       "page_start": page_start, "page_end": page_end, **extra})

    def flush_text():
        nonlocal text_buf
        if text_buf:
            emit("text", "\n\n".join(t for t, _ in text_buf), text_buf[0][1], text_buf[-1][1])
            text_buf = []

    def emit_table(rows: list[str], page: int):
        nonlocal n_tables
        has_header = len(rows) >= 2 and _is_sep(rows[1])
        hdr_rows, body = (rows[:2], rows[2:]) if has_header else (st["table_header"] or [], rows)
        given, last = st["table_title"], st["last_table_title"]
        same_cols = bool(hdr_rows and st["table_header"]) and _cells(hdr_rows[0]) == _cells(st["table_header"][0])
        continued = st["table_id"] is not None and (
            not has_header or given == "(continued)" or (given in (None, last) and same_cols))
        note = st["pending_note"] or (st["last_para"] if st["prev_kind"] == "para" else None)
        if note and n_tokens(note) > TABLE_NOTE_MAX_TOKENS:
            note = None
        if continued:
            title = last
            if note and not st["table_note"]:
                st["table_note"] = note
        else:
            title = given if given not in (None, "(continued)") else (st["section"] or "Table")
            n_tables += 1
            st["table_id"], st["table_note"] = f"t{n_tables:03d}", note
        st["table_header"], st["last_table_title"], st["pending_note"] = hdr_rows, title, None

        # split into groups at group-label rows, then pack groups up to the token budget
        groups, cur = [], []
        for r in body:
            if _is_group_row(r) and cur:
                groups.append(cur); cur = []
            cur.append(r)
        if cur:
            groups.append(cur)
        packs, cur, cur_tok = [], [], 0
        for g in groups:
            for sub in [g[i:i + 40] for i in range(0, len(g), 40)]:   # cap pathological groups
                t = sum(n_tokens(r) for r in sub)
                if cur and cur_tok + t > TABLE_CHUNK_TOKENS:
                    packs.append(cur); cur, cur_tok = [], 0
                cur += sub; cur_tok += t
        if cur:
            packs.append(cur)

        for rows_ in packs:
            groups_in = [_cells(r)[0] for r in rows_ if _is_group_row(r)]
            body_md = "\n".join(hdr_rows + rows_)
            note = f"Table note: {st['table_note']}\n" if st["table_note"] else ""
            emit("table", f"{note}{body_md}", page, table_id=st["table_id"], table_title=title,
                 table_groups=groups_in, table_header_md="\n".join(hdr_rows), table_rows_md="\n".join(rows_))

    for p in pages:
        for b in parse_blocks(p["markdown"]):
            kind = b[0]
            if kind == "heading":
                flush_text()
                level, title = b[1], b[2]
                if title.casefold() == (st["section"] or "").casefold():
                    pass                                        # section name repeated on a continuation page
                elif level <= 2:
                    st.update(section=title, subsection=None, table_note=None, table_title=None, table_id=None)
                else:
                    st["subsection"] = title
            elif kind == "para" and (pm := _PIN_RE.match(b[1])):
                flush_text()                                   # one chunk per pin description
                emit("pin", b[1], p["page"], pin_name=pm.group(1).strip())
                st["last_para"] = None
            elif kind == "para":
                para = b[1]
                if text_buf and sum(n_tokens(t) for t, _ in text_buf) + n_tokens(para) > TEXT_CHUNK_TOKENS:
                    flush_text()
                text_buf.append((para, p["page"]))
                st["last_para"] = para
            elif kind == "table_title":
                st["table_title"] = b[1] or "(continued)"
                if st["prev_kind"] == "para":
                    st["pending_note"] = st["last_para"]
            elif kind == "table":
                flush_text()
                emit_table(b[1], p["page"])
                st["table_title"] = None
            elif kind == "figure":
                flush_text()
                m = re.search(r"Title:\s*(.+)", b[1])
                ftype = re.search(r"Type:\s*(.+)", b[1])
                emit("figure", b[1], p["page"], figure_title=m.group(1).strip() if m else None,
                     figure_type=ftype.group(1).strip() if ftype else None)
            st["prev_kind"] = kind
    flush_text()

    for i, c in enumerate(chunks):
        c["chunk_index"] = i
    # page-level points
    for p in pages:
        md_text = p["markdown"]
        toks = enc.encode(md_text)
        if len(toks) > PAGE_MAX_TOKENS:
            md_text = enc.decode(toks[:PAGE_MAX_TOKENS])
        chunks.append({"content_type": "page", "text": f"[{part} · p.{p['page']}]\n{md_text}", "content": p["markdown"],
                       "page_start": p["page"], "page_end": p["page"], "chunk_index": -p["page"],
                       "parse_method": p["method"]})
    return chunks

## 6. Vectors: dense (OpenAI) + sparse (BM25)

The sparse vectors are BM25 term weights computed locally (`datasheet_rag/sparse.py`, shared with the retrieval side so documents and queries are tokenized identically). The tokenizer keeps datasheet identifiers whole (`ltc3310hv-1#pbf`, `rds(on)`, `3.3v`) and also indexes their parts. Qdrant applies IDF on the server (`Modifier.IDF`), so the weights stay correct as more documents are added.

In [ ]:
from datasheet_rag.sparse import tokenize, sparse_doc

def embed_texts(texts: list[str]) -> list[list[float]]:
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        resp = oai.embeddings.create(model=EMBED_MODEL, input=texts[i:i + EMBED_BATCH])
        out += [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]
    return out

## 7. Qdrant collection (named dense + sparse vectors, payload indexes)

In [ ]:
RECREATE = False    # True = wipe and rebuild

if RECREATE and qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

if not qdrant.collection_exists(COLLECTION):
    qdrant.create_collection(
        COLLECTION,
        vectors_config={"dense": models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE)},
        sparse_vectors_config={"bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    print(f"Created '{COLLECTION}'")

for field, schema in [("source", "keyword"), ("doc_id", "keyword"), ("content_type", "keyword"),
                      ("section", "keyword"), ("table_id", "keyword"), ("part_numbers", "keyword"), ("pin_name", "keyword"),
                      ("page_start", "integer"), ("chunk_index", "integer")]:
    qdrant.create_payload_index(COLLECTION, field_name=field, field_schema=schema)

## 8. Ingest (idempotent: unchanged files are skipped, changed files are replaced)

In [ ]:
NAMESPACE = uuid.UUID("6f1c7a52-2b0e-4b8e-9a77-3d1f4c0b9e11")

def _match(key, value):
    return models.FieldCondition(key=key, match=models.MatchValue(value=value))

def ingest_pdf(pdf_path: Path) -> list[dict]:
    # Root span = one trace per PDF in Phoenix; metadata/tags propagate to every child (incl. auto-traced OpenAI calls)
    with using_attributes(metadata={"source": pdf_path.name, "collection": COLLECTION}, tags=["ingestion"]), \
         tracer.start_as_current_span("ingest_pdf", openinference_span_kind="chain") as root:
        root.set_input(str(pdf_path))
        try:
            chunks = _ingest_pdf(pdf_path, root)
            root.set_status(Status(StatusCode.OK))
            return chunks
        except Exception as e:
            root.record_exception(e)
            root.set_status(Status(StatusCode.ERROR, str(e)))
            raise

def _ingest_pdf(pdf_path: Path, root) -> list[dict]:
    doc_id = file_sha256(pdf_path)
    root.set_attributes({"doc.id": doc_id, "doc.source": pdf_path.name, "doc.bytes": pdf_path.stat().st_size})
    if qdrant.count(COLLECTION, count_filter=models.Filter(must=[_match("doc_id", doc_id)]), exact=True).count:
        print(f"• {pdf_path.name}: already ingested, skipping")
        root.set_attribute("ingest.skipped", True)
        root.set_output("skipped: unchanged file already in collection")
        return []

    pages = parse_pdf(pdf_path, doc_id)

    with tracer.start_as_current_span("extract_doc_meta", openinference_span_kind="chain") as span:
        meta = extract_doc_meta(pages)
        span.set_output(as_json(meta), mime_type="application/json")

    with tracer.start_as_current_span("build_chunks", openinference_span_kind="chain") as span:
        chunks = build_chunks(pages, meta, pdf_path.name)
        by_type = Counter(c["content_type"] for c in chunks)
        toks = [n_tokens(c["text"]) for c in chunks]
        span.set_attributes({f"chunks.{k}": v for k, v in by_type.items()} |
                            {"chunks.total": len(chunks), "chunks.tokens_max": max(toks),
                             "chunks.tokens_mean": round(sum(toks) / len(toks), 1),
                             "chunks.tables_distinct": len({c.get("table_id") for c in chunks if c.get("table_id")})})
        span.set_output(as_json([{k: c.get(k) for k in ("content_type", "section", "page_start", "table_id", "pin_name")}
                                 | {"preview": c["text"][:200]} for c in chunks]), mime_type="application/json")
    print(f"• {meta['title']}\n  parts={meta['part_numbers']}  pages={len(pages)}  chunks={dict(by_type)}")

    with tracer.start_as_current_span("embed_chunks", openinference_span_kind="chain") as span:
        span.set_attributes({"embedding.model_name": EMBED_MODEL, "embedding.count": len(chunks), "embedding.total_tokens": sum(toks)})
        dense = embed_texts([c["text"] for c in tqdm(chunks, desc="Embedding", leave=False)])
    now   = datetime.now(timezone.utc).isoformat()
    points = [models.PointStruct(
        id=str(uuid.uuid5(NAMESPACE, f"{doc_id}:{c['content_type']}:{c['chunk_index']}")),
        vector={"dense": vec, "bm25": sparse_doc(c["text"])},
        payload={**c, "source": pdf_path.name, "source_path": str(pdf_path.resolve()), "doc_id": doc_id, "title": meta["title"],
                 "manufacturer": meta["manufacturer"], "part_numbers": meta["part_numbers"],
                 "device_category": meta["device_category"], "doc_summary": meta["summary"],
                 "num_pages": len(pages), "parse_model": PARSE_MODEL, "embedding_model": EMBED_MODEL,
                 "ingested_at": now},
    ) for c, vec in zip(chunks, dense)]

    with tracer.start_as_current_span("qdrant_upsert", openinference_span_kind="chain") as span:
        span.set_attributes({"db.system": "qdrant", "db.collection": COLLECTION, "db.points": len(points)})
        qdrant.delete(COLLECTION, points_selector=models.FilterSelector(       # drop older versions of this file
            filter=models.Filter(must=[_match("source", pdf_path.name)])))
        for i in range(0, len(points), 128):
            qdrant.upsert(COLLECTION, points=points[i:i + 128], wait=True)

    root.set_attributes({"doc.title": meta["title"], "doc.pages": len(pages), "ingest.points": len(points)})
    root.set_output(as_json({"title": meta["title"], "pages": len(pages), "chunks": dict(by_type)}), mime_type="application/json")
    return chunks

all_chunks = {}
for pdf in PDF_PATHS:
    all_chunks[pdf.name] = ingest_pdf(pdf)
tracer_provider.force_flush()
print("Points in collection:", qdrant.count(COLLECTION, exact=True).count)
print(f"Traces: {config.PHOENIX_ENDPOINT}  (project '{PHOENIX_PROJECT}')")

### Inspect what was produced

In [ ]:
for name, chs in all_chunks.items():
    for c in [c for c in chs if c["content_type"] == "table"][:2] + [c for c in chs if c["content_type"] == "figure"][:1]:
        print("=" * 100); print(c["text"][:1500])

## 9. Retrieval functions

These live in `datasheet_rag/retrieval.py` and are shared with the agent notebook (`agentic_rag.ipynb`), which wraps them as agent tools.

| Function | Purpose |
|---|---|
| `search_datasheets(query, top_k, content_types, sources, part_number)` | Hybrid dense + BM25 search with RRF fusion, plus filters (RETRIEVER span) |
| `get_table(doc_id, table_id)` | Rebuild a full table from its chunks, when a hit is one slice of a large table |
| `get_page(doc_id, page)` | Full Markdown of a page, for the surrounding context |
| `get_neighbor_chunks(doc_id, chunk_index, window)` | Text immediately before and after a hit |
| `list_documents()` | What the knowledge base contains |
| `render_page_image(doc_id, page)` | PNG of the original PDF page (uses the stored `source_path`) |

In [ ]:
from datasheet_rag.retrieval import (search_datasheets, get_table, get_page, get_neighbor_chunks,
                                     list_documents, render_page_image)

## 10. Retrieval sanity checks

In [ ]:
for d in list_documents():
    print(d["title"], "|", d["part_numbers"], "|", d["num_pages"], "pages")

QUESTIONS = [
    "What is the VIN undervoltage lockout threshold and hysteresis?",
    "PMOS switch on-resistance at 5V input",
    "What is the part marking for LTC3310HV-1#PBF?",
    "What is the absolute maximum junction temperature?",
    "What does the PGOOD pin do?",
    "How do I choose the inductor value?",
]
for q in QUESTIONS:
    print("\n" + "#" * 100 + f"\nQ: {q}")
    for h in search_datasheets(q, top_k=3):
        print(f"  [{h['score']}] {h['content_type']:6s} p.{h['page_start']} {h['section']} {h['table_title'] or h['figure_title'] or ''}")
    print("  top hit:\n   ", search_datasheets(q, top_k=1)[0]["text"][:700].replace("\n", "\n    "))

# exercise the expansion tools too, so they show up in Phoenix
hit = search_datasheets("electrical characteristics oscillator frequency", top_k=1, content_types=["table"])[0]
_ = get_table(hit["doc_id"], hit["table_id"])
_ = get_neighbor_chunks(hit["doc_id"], hit["chunk_index"])
print("page image bytes:", len(render_page_image(hit["doc_id"], hit["page_start"])))
tracer_provider.force_flush()

## 11. Using the traces in Phoenix

Open http://localhost:6006 and go to project **datasheet-rag**.

| What to check | Where |
|---|---|
| Per-page parse quality | `ingest_pdf` trace → `parse_page N` → output Markdown. The child LLM span has the prompt, text layer, and token counts. Set `TRACE_PAGE_IMAGES = True` to see the page image alongside. |
| Slow or expensive pages | Sort `parse_page` spans by latency, or the LLM spans by token count |
| Vision failures and fallbacks | Filter spans with `attributes["parse.method"] == "pymupdf4llm"`, or spans with status ERROR |
| Chunking stats | `build_chunks` span attributes (counts per type, token max/mean) and the chunk preview list |
| Retrieval quality | RETRIEVER spans (`search_datasheets`): the query plus the ranked documents with scores. Filter with `span_kind == 'RETRIEVER'`. |

**Next:** `agentic_rag.ipynb` answers questions over this collection with an agent. Its traces go to the separate Phoenix project `datasheet-agent`, where each question is one trace (agent → LLM steps → tool calls → retrieval → verifier).